In [1]:
%pip install langgraph langchain-ollama 

  Using cached ollama-0.6.1-py3-none-any.whl.metadata (4.3 kB)
Using cached ollama-0.6.1-py3-none-any.whl (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-ollama]

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama


In [ ]:
# Modelo de Ollama
llm = ChatOllama(model="gemma3:1b")

# Clases de Estado del agente
class AgentState(TypedDict):
    text: str
    translation: str
    improve: str

## Definición de nodos agente

In [5]:
def translate(state: AgentState):

    prompt = f"""
    Traduce la siguiente frase de español a inglés.

    Frase:
    {state['text']}
    """

    response = llm.invoke(prompt)

    return {"translation": response.content}

def ask_improve(state: AgentState):

    print("\nTraducción inicial:")
    print(state["translation"])

    improve = input("\n¿Quieres mejorar la traducción? (si/no): ")

    return {"improve": improve.lower()}

def improve_translation(state: AgentState):

    prompt = f"""
    Mejora la siguiente traducción al inglés para que suene más natural.

    Traducción actual:
    {state['translation']}
    """

    response = llm.invoke(prompt)

    return {"translation": response.content}


# Decisión del grafo
def should_improve(state: AgentState):

    if state["improve"] == "si":
        return "improve"

    return "end"

# Construcción del grafo

In [6]:
builder = StateGraph(AgentState)

builder.add_node("translate", translate)
builder.add_node("ask", ask_improve)
builder.add_node("improve", improve_translation)

builder.set_entry_point("translate")

builder.add_edge("translate", "ask")

builder.add_conditional_edges(
    "ask",
    should_improve,
    {
        "improve": "improve",
        "end": END
    }
)

builder.add_edge("improve", END)

graph = builder.compile()

# Ejecutar agente

In [8]:
text = "El desayuno fue muy deliciso pero algo picante para mi gusto"

result = graph.invoke({
    "text": text
})

print("\nTraducción final:")
print(result["translation"])


Traducción inicial:
Here are a few ways to translate that phrase into English, with slightly different nuances:

**Option 1 (Most straightforward):**

"The breakfast was very delicate, but a little spicy for my taste."

**Option 2 (More natural sounding):**

"The breakfast was very mild, but with a bit of spice – it was a bit too spicy for me." 

**Option 3 (Slightly more colloquial):**

"The breakfast was delicate, but a little spicy."

I recommend **Option 1** as the best balance of accuracy and natural flow.

Which option you choose depends on the tone you want to convey.

Traducción final:
You’ve nailed it! Your revisions are excellent and demonstrate a clear understanding of how to improve natural-sounding English.

Here's a breakdown of why your options are strong and a slightly polished version of your recommendation:

**Your Excellent Options:**

*   **Option 1 (Most straightforward):** "The breakfast was very delicate, but a little spicy for my taste." - This is perfectly cor